In [1]:
from ragwire import RAGWire, setup_logging
import ragwire

logger = setup_logging(log_level="INFO")

print(ragwire.__version__)

rag = RAGWire("config.yaml")
results = rag.retrieve("What is Apple's revenue in 2025?", top_k=3)

for doc in results:
    print(doc.metadata)

1.2.7
2026-03-26 10:25:22,949 - ragwire.core.pipeline - INFO - Loading configuration from config.yaml
2026-03-26 10:25:23,388 - ragwire.core.pipeline - INFO - Document loader initialized
2026-03-26 10:25:23,389 - ragwire.core.pipeline - INFO - Text splitter initialized (strategy=markdown, chunk_size=10000)
2026-03-26 10:25:23,804 - ragwire.core.pipeline - INFO - Embedding model initialized (provider=ollama)
2026-03-26 10:25:23,825 - ragwire.core.pipeline - INFO - LLM initialized for metadata extraction (provider=ollama, model=qwen3.5:4b)
2026-03-26 10:25:24,392 - ragwire.vectorstores.qdrant_store - INFO - Connected to Qdrant at http://192.168.1.9:6333
2026-03-26 10:25:34,422 - ragwire.core.pipeline - INFO - Using existing collection: financial_docs
2026-03-26 10:25:34,755 - ragwire.core.pipeline - INFO - Vector store initialized
2026-03-26 10:25:34,756 - ragwire.core.pipeline - INFO - Retriever initialized (type=hybrid, top_k=5, auto_filter=False)
2026-03-26 10:25:34,756 - ragwire.core

In [2]:
rag._auto_filter = True
# rag._auto_filter = False
rag._auto_filter

True

In [3]:
results = rag.retrieve("What is Apple's revenue in 2025?", top_k=3)

for doc in results:
    print(doc.metadata)

2026-03-26 10:25:37,022 - ragwire.core.pipeline - INFO - Auto-extracted filters from query: {'company_name': 'apple inc', 'fiscal_year': 2025}
2026-03-26 10:25:37,133 - ragwire.core.pipeline - INFO - Retrieved 3 documents for query: What is Apple's revenue in 2025?...
{'source': 'data/Apple_10k_2025.pdf', 'file_name': 'Apple_10k_2025.pdf', 'file_type': 'pdf', 'file_hash': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab', 'chunk_id': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab_16', 'chunk_hash': '60b8d552baec48866282206f62d1d54dad52bc158891d1dcf5f9849c24cf4082', 'chunk_index': 16, 'total_chunks': 35, 'created_at': '2026-03-23T18:25:14.303535+00:00', 'company_name': 'apple inc', 'doc_type': '10-k', 'fiscal_quarter': None, 'fiscal_year': [2025], '_id': 'd0610359-6e3f-467d-aaa4-9893c953fb68', '_collection_name': 'financial_docs'}
{'source': 'data/Apple_10k_2025.pdf', 'file_name': 'Apple_10k_2025.pdf', 'file_type': 'pdf', 'file_hash': '108590052c3ba54

In [4]:
rag.discover_metadata_fields()
# rag.get_field_values("company_name")
# rag.get_field_values(["company_name", "doc_type"])
# rag.get_field_values("file_name", limit=200)


['source',
 'file_name',
 'file_type',
 'file_hash',
 'chunk_id',
 'chunk_hash',
 'chunk_index',
 'total_chunks',
 'created_at',
 'company_name',
 'doc_type',
 'fiscal_quarter',
 'fiscal_year']

In [5]:
filter_context = rag.get_filter_context("What is Apple's revenue in 2025?")
print(filter_context)

2026-03-26 10:26:04,483 - ragwire.core.pipeline - INFO - Auto-extracted filters from query: {'company_name': 'apple inc', 'fiscal_year': 2025}
## RAGWire Filter Context

### Available Metadata Fields and Stored Values
- **company_name**: ['apple inc']
- **doc_type**: ['10-k']
- **fiscal_quarter**: []
- **fiscal_year**: [2025]

### Extracted Filters from Query
- **company_name**: `apple inc`
- **fiscal_year**: `2025`

### Instructions
1. Review the extracted filters above.
2. If an extracted value does not match or closely relate to any stored value, adjust or drop that filter.
3. If the query has no clear metadata intent, pass an empty dict `{}` as filters.
4. Pass the final filters dict to the retrieval tool as `filters=`.


In [6]:
results = rag.retrieve(
    "What is the total revenue?",
    top_k=5,
    filters={"company_name": "apple inc"}
)
results

2026-03-26 10:26:28,148 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: What is the total revenue?...


[Document(metadata={'source': 'data/Apple_10k_2025.pdf', 'file_name': 'Apple_10k_2025.pdf', 'file_type': 'pdf', 'file_hash': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab', 'chunk_id': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab_16', 'chunk_hash': '60b8d552baec48866282206f62d1d54dad52bc158891d1dcf5f9849c24cf4082', 'chunk_index': 16, 'total_chunks': 35, 'created_at': '2026-03-23T18:25:14.303535+00:00', 'company_name': 'apple inc', 'doc_type': '10-k', 'fiscal_quarter': None, 'fiscal_year': [2025], '_id': 'd0610359-6e3f-467d-aaa4-9893c953fb68', '_collection_name': 'financial_docs'}, page_content='Earnings per share:\n\nBasic\nDiluted\n\nShares used in computing earnings per share:\n\nBasic\n\nDiluted\n\nSeptember 27,\n2025\n\nYears ended\n\nSeptember 28,\n2024\n\nSeptember 30,\n2023\n\n$\n\n$\n\n$\n$\n\n307,003  $\n109,158\n\n416,161\n\n294,866  $\n96,169\n\n391,035\n\n194,116\n26,844\n\n220,960\n\n195,201\n\n34,550\n27,601\n62,151\n\n185,233\n25,

In [7]:
results = rag.retrieve(
    "What are the risk factors?",
    top_k=5,
    filters={"doc_type": "10-k"}
)
results

2026-03-26 10:26:32,220 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: What are the risk factors?...


[Document(metadata={'source': 'data/Apple_10k_2025.pdf', 'file_name': 'Apple_10k_2025.pdf', 'file_type': 'pdf', 'file_hash': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab', 'chunk_id': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab_3', 'chunk_hash': '1185412ef5dc739f6bbf7913f64b3996a869f45cbdf3564a44864ee95a554df4', 'chunk_index': 3, 'total_chunks': 35, 'created_at': '2026-03-23T18:25:14.303329+00:00', 'company_name': 'apple inc', 'doc_type': '10-k', 'fiscal_quarter': None, 'fiscal_year': [2025], '_id': 'b944bbae-3a9e-4636-b4c4-7fa7027cc8c5', '_collection_name': 'financial_docs'}, page_content='Apple Inc. | 2025 Form 10-K | 4\n\n\x0cItem 1A.    Risk Factors\n\nThe following summarizes factors that could have a material adverse effect on the Company’s business, reputation, results of operations, financial condition and\nstock price. The Company may not be able to accurately predict, control or mitigate these risks. Statements in this section are ba

In [8]:
# Single year — pass as int
results = rag.retrieve(
    "What is the net income?",
    top_k=5,
    filters={"fiscal_year": 2025}
)
print(results)

# Multiple years — pass as list; matches documents covering ANY of the years (OR logic)
results = rag.retrieve(
    "Compare net income across 2023 and 2024",
    top_k=10,
    filters={"fiscal_year": [2023, 2024]}
)
print(results)

2026-03-26 10:26:33,218 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: What is the net income?...
[Document(metadata={'source': 'data/Apple_10k_2025.pdf', 'file_name': 'Apple_10k_2025.pdf', 'file_type': 'pdf', 'file_hash': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab', 'chunk_id': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab_15', 'chunk_hash': '5749489e69466a17a5a6e9b9a7d17f5a1a4f3c01e7836ebce3310f6a278e0fdb', 'chunk_index': 15, 'total_chunks': 35, 'created_at': '2026-03-23T18:25:14.303519+00:00', 'company_name': 'apple inc', 'doc_type': '10-k', 'fiscal_quarter': None, 'fiscal_year': [2025], '_id': '01ee9bf4-70b3-4d40-abc2-c46fb8908ad7', '_collection_name': 'financial_docs'}, page_content='Income Taxes\n\nIn December 2023, the FASB issued ASU No. 2023-09, Income Taxes (Topic 740): Improvements to Income Tax Disclosures (“ASU 2023-09”), which will require\nthe Company to disclose specified additional information in its income 

In [9]:
results = rag.retrieve(
    "What is the revenue breakdown by segment?",
    top_k=5,
    filters={
        "company_name": "apple inc",
        "fiscal_year": 2025
    }
)
results

2026-03-26 10:26:37,524 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: What is the revenue breakdown by segment?...


[Document(metadata={'source': 'data/Apple_10k_2025.pdf', 'file_name': 'Apple_10k_2025.pdf', 'file_type': 'pdf', 'file_hash': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab', 'chunk_id': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab_17', 'chunk_hash': '6f8d154ee3123334dcf4f277a15aaf1d235ed46a0a4fa4a3adcf44b3ade8c2b6', 'chunk_index': 17, 'total_chunks': 35, 'created_at': '2026-03-23T18:25:14.303550+00:00', 'company_name': 'apple inc', 'doc_type': '10-k', 'fiscal_quarter': None, 'fiscal_year': [2025], '_id': 'a439567e-7706-47ec-b8bb-d70ae36e725f', '_collection_name': 'financial_docs'}, page_content='Recently Adopted Accounting Pronouncements\n\nSegment Reporting\n\nBeginning  with  the  2025  annual  reporting  period,  the  Company  adopted  the  FASB’s  ASU  No.  2023-07,  Segment  Reporting  (Topic  280):  Improvements  to\nReportable Segment Disclosures (“ASU 2023-07”), which requires the Company to disclose segment expenses that are significant 

In [10]:
results = rag.hybrid_search(
    "Apple revenue fiscal 2025",
    k=5,
    filters={"company_name": "apple inc"}
)
results

[Document(metadata={'source': 'data/Apple_10k_2025.pdf', 'file_name': 'Apple_10k_2025.pdf', 'file_type': 'pdf', 'file_hash': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab', 'chunk_id': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab_16', 'chunk_hash': '60b8d552baec48866282206f62d1d54dad52bc158891d1dcf5f9849c24cf4082', 'chunk_index': 16, 'total_chunks': 35, 'created_at': '2026-03-23T18:25:14.303535+00:00', 'company_name': 'apple inc', 'doc_type': '10-k', 'fiscal_quarter': None, 'fiscal_year': [2025], '_id': 'd0610359-6e3f-467d-aaa4-9893c953fb68', '_collection_name': 'financial_docs'}, page_content='Earnings per share:\n\nBasic\nDiluted\n\nShares used in computing earnings per share:\n\nBasic\n\nDiluted\n\nSeptember 27,\n2025\n\nYears ended\n\nSeptember 28,\n2024\n\nSeptember 30,\n2023\n\n$\n\n$\n\n$\n$\n\n307,003  $\n109,158\n\n416,161\n\n294,866  $\n96,169\n\n391,035\n\n194,116\n26,844\n\n220,960\n\n195,201\n\n34,550\n27,601\n62,151\n\n185,233\n25,

In [11]:
from ragwire import RAGWire

rag = RAGWire("config.yaml")

# filter_fields returns only semantic/filterable fields (e.g. company_name, doc_type)
# Use this instead of discover_metadata_fields() which includes system fields
# like file_hash, chunk_id, source that are not useful for filtering
fields = rag.filter_fields
values = rag.get_field_values(fields)

field_descriptions = "\n".join(
    f"- {field}: {values[field]}" if values.get(field) else f"- {field}"
    for field in fields
)

SYSTEM_PROMPT = f"""
You are a document assistant with access to a RAG pipeline.

Available metadata fields and known values:
{field_descriptions}

When answering questions, extract any filters from the user query and apply them.
"""

print(SYSTEM_PROMPT)

2026-03-26 10:26:40,478 - ragwire.core.pipeline - INFO - Loading configuration from config.yaml
2026-03-26 10:26:40,531 - ragwire.core.pipeline - INFO - Document loader initialized
2026-03-26 10:26:40,532 - ragwire.core.pipeline - INFO - Text splitter initialized (strategy=markdown, chunk_size=10000)
2026-03-26 10:26:40,570 - ragwire.core.pipeline - INFO - Embedding model initialized (provider=ollama)
2026-03-26 10:26:40,592 - ragwire.core.pipeline - INFO - LLM initialized for metadata extraction (provider=ollama, model=qwen3.5:4b)
2026-03-26 10:26:40,618 - ragwire.vectorstores.qdrant_store - INFO - Connected to Qdrant at http://192.168.1.9:6333
2026-03-26 10:26:40,656 - ragwire.core.pipeline - INFO - Using existing collection: financial_docs
2026-03-26 10:26:40,950 - ragwire.core.pipeline - INFO - Vector store initialized
2026-03-26 10:26:40,952 - ragwire.core.pipeline - INFO - Retriever initialized (type=hybrid, top_k=5, auto_filter=False)
2026-03-26 10:26:40,954 - ragwire.core.pipel

#### Full example — LLM-driven filter extraction¶


In [12]:
from ragwire import RAGWire
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
import json

rag = RAGWire("config.yaml")
llm = ChatOllama(model="nemotron-3-nano:4b", temperature=0, base_url="http://192.168.1.9:11434")

# filter_fields returns only the semantic fields used for filtering
# (excludes system fields like file_hash, chunk_id, source, created_at)
fields = rag.filter_fields
values = rag.get_field_values(fields)

# Build the filter prompt dynamically — works for any schema
field_descriptions = "\n".join(
    f"- {field}: {values[field]}" if values.get(field) else f"- {field}"
    for field in fields
)

FILTER_PROMPT = f"""
Given the user query below, extract metadata filters as JSON.
Only include fields if clearly mentioned in the query. Return {{{{}}}} if no filters apply.

Available metadata fields and known values:
{field_descriptions}

User query: {{query}}

Filters (JSON only):
"""

def query_with_filters(user_query: str, top_k: int = 5):
    # Step 1: LLM extracts filters from the query using the dynamic prompt
    prompt = ChatPromptTemplate.from_template(FILTER_PROMPT)
    chain = prompt | llm
    response = chain.invoke({"query": user_query})

    try:
        filters = json.loads(response.content.strip())
    except Exception:
        filters = {}

    print(f"Extracted filters: {filters}")

    # Step 2: Retrieve with extracted filters
    results = rag.retrieve(user_query, top_k=top_k, filters=filters or None)
    return results

2026-03-26 10:26:52,675 - ragwire.core.pipeline - INFO - Loading configuration from config.yaml
2026-03-26 10:26:52,728 - ragwire.core.pipeline - INFO - Document loader initialized
2026-03-26 10:26:52,729 - ragwire.core.pipeline - INFO - Text splitter initialized (strategy=markdown, chunk_size=10000)
2026-03-26 10:26:52,878 - ragwire.core.pipeline - INFO - Embedding model initialized (provider=ollama)
2026-03-26 10:26:52,917 - ragwire.core.pipeline - INFO - LLM initialized for metadata extraction (provider=ollama, model=qwen3.5:4b)
2026-03-26 10:26:52,938 - ragwire.vectorstores.qdrant_store - INFO - Connected to Qdrant at http://192.168.1.9:6333
2026-03-26 10:26:52,958 - ragwire.core.pipeline - INFO - Using existing collection: financial_docs
2026-03-26 10:26:53,245 - ragwire.core.pipeline - INFO - Vector store initialized
2026-03-26 10:26:53,246 - ragwire.core.pipeline - INFO - Retriever initialized (type=hybrid, top_k=5, auto_filter=False)
2026-03-26 10:26:53,247 - ragwire.core.pipel

In [13]:
query_with_filters("What is Apple's revenue for fiscal year 2025?")

Extracted filters: {'company_name': 'apple inc', 'fiscal_year': 2025}
2026-03-26 10:26:57,814 - ragwire.core.pipeline - INFO - Retrieved 5 documents for query: What is Apple's revenue for fiscal year 2025?...


[Document(metadata={'source': 'data/Apple_10k_2025.pdf', 'file_name': 'Apple_10k_2025.pdf', 'file_type': 'pdf', 'file_hash': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab', 'chunk_id': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab_16', 'chunk_hash': '60b8d552baec48866282206f62d1d54dad52bc158891d1dcf5f9849c24cf4082', 'chunk_index': 16, 'total_chunks': 35, 'created_at': '2026-03-23T18:25:14.303535+00:00', 'company_name': 'apple inc', 'doc_type': '10-k', 'fiscal_quarter': None, 'fiscal_year': [2025], '_id': 'd0610359-6e3f-467d-aaa4-9893c953fb68', '_collection_name': 'financial_docs'}, page_content='Earnings per share:\n\nBasic\nDiluted\n\nShares used in computing earnings per share:\n\nBasic\n\nDiluted\n\nSeptember 27,\n2025\n\nYears ended\n\nSeptember 28,\n2024\n\nSeptember 30,\n2023\n\n$\n\n$\n\n$\n$\n\n307,003  $\n109,158\n\n416,161\n\n294,866  $\n96,169\n\n391,035\n\n194,116\n26,844\n\n220,960\n\n195,201\n\n34,550\n27,601\n62,151\n\n185,233\n25,